In [0]:
CREATE SCHEMA IF NOT EXISTS olist_maas_pipeline.silver;

In [0]:

-- STEP 1: Sanitisation
-- Fix review_score (Nullifying '0' values)
CREATE OR REPLACE TABLE olist_maas_pipeline.silver.sales_order_reviews AS
SELECT review_id
  , order_id
  , CASE WHEN review_score < 1 THEN NULL 
   ELSE review_score END AS review_score
  , review_comment_title
  , review_comment_message
  , review_creation_date
  , review_answer_timestamp
FROM olist_maas_pipeline.bronze.sales_order_reviews;

-- Fix null marketing origin
CREATE OR REPLACE TABLE olist_maas_pipeline.silver.marketing_mql AS
SELECT mql_id
  , first_contact_date
  , landing_page_id
  , COALESCE(origin, "unknown") AS origin
FROM olist_maas_pipeline.bronze.marketing_mql;

In [0]:
-- STEP 2: Infrastructure Tiering & Breaking Point Mapping
CREATE OR REPLACE TABLE olist_maas_pipeline.silver.dim_logistics_regions AS
WITH tier_thresholds AS (
  SELECT * FROM (VALUES
    (1, 5, 'High urban infrastructure')
    , (2, 7, 'Standard regional infrastructure')
    , (3, 10, 'Complex/Sparse infrastructure')
  ) AS t(tier, breaking_point_days, tier_description)
),
state_to_region AS (
  SELECT * FROM (VALUES
    ('SP', 'Southeast', 1), ('RJ', 'Southeast', 1), ('MG', 'Southeast', 1), ('ES', 'Southeast', 1)
    , ('PR', 'South', 2), ('SC', 'South', 2), ('RS', 'South', 2)
    , ('GO', 'Central-West', 2), ('DF', 'Central-West', 2), ('MT', 'Central-West', 2), ('MS', 'Central-West', 2)
    , ('AM', 'North', 3), ('PA', 'North', 3), ('AC', 'North', 3), ('RO', 'North', 3)
    , ('RR', 'North', 3), ('AP', 'North', 3), ('TO', 'North', 3), ('BA', 'Northeast', 3)
    , ('PE', 'Northeast', 3), ('CE', 'Northeast', 3), ('MA', 'Northeast', 3), ('PI', 'Northeast', 3)
    , ('RN', 'Northeast', 3), ('PB', 'Northeast', 3), ('AL', 'Northeast', 3), ('SE', 'Northeast', 3)
  ) AS t(state_code, region_name, tier_id)
)
SELECT 
    s.state_code
    , s.region_name
    , s.tier_id AS logistics_tier
    , t.breaking_point_days
    , t.tier_description
FROM state_to_region s
JOIN tier_thresholds t ON s.tier_id = t.tier;

In [0]:
-- STEP 3: Enriched fact and dimension tables
-- Enriched orders
CREATE OR REPLACE TABLE olist_maas_pipeline.silver.sales_orders AS
SELECT 
    o.order_id
    , o.customer_id
    , c.customer_unique_id
    , c.customer_state
    , l.region_name
    , l.logistics_tier
    , l.breaking_point_days
    , o.order_status
    , TO_TIMESTAMP(o.order_purchase_timestamp) AS order_purchase_timestamp
    , TO_TIMESTAMP(o.order_delivered_customer_date) AS order_delivered_customer_date
    , TO_TIMESTAMP(o.order_estimated_delivery_date) AS order_estimated_delivery_date
    , DATEDIFF(
        TO_TIMESTAMP(o.order_delivered_customer_date)
        , TO_TIMESTAMP(o.order_purchase_timestamp)
      ) AS actual_delivery_days
FROM olist_maas_pipeline.bronze.sales_orders o
LEFT JOIN olist_maas_pipeline.bronze.sales_customers c 
    ON o.customer_id = c.customer_id
LEFT JOIN olist_maas_pipeline.silver.dim_logistics_regions l 
    ON c.customer_state = l.state_code;

-- Enriched order items
-- Contains seller_id & shipping_limit_date to handle multi-seller orders
CREATE OR REPLACE TABLE olist_maas_pipeline.silver.sales_order_items AS
SELECT 
    i.order_id
    , i.order_item_id
    , i.product_id
    , i.seller_id
    , i.price 
    , i.shipping_limit_date
    , i.freight_value 
    , c.product_category_name_english AS product_category_name
    , p.product_weight_g
    , p.product_length_cm
    , p.product_height_cm
    , p.product_width_cm
FROM olist_maas_pipeline.bronze.sales_order_items i
LEFT JOIN olist_maas_pipeline.bronze.sales_products p 
    ON i.product_id = p.product_id
LEFT JOIN olist_maas_pipeline.bronze.sales_product_category_translation c
    ON p.product_category_name = c.product_category_name;

-- Enriched order payments
CREATE OR REPLACE TABLE olist_maas_pipeline.silver.sales_order_payments AS
SELECT 
    order_id
    , payment_sequential
    , payment_type
    , payment_installments
    , payment_value
FROM olist_maas_pipeline.bronze.sales_order_payments;

-- sellers master
CREATE OR REPLACE TABLE olist_maas_pipeline.silver.sales_sellers AS
SELECT 
    s.seller_id
    , s.seller_zip_code_prefix
    , s.seller_city
    , s.seller_state
    , l.region_name AS seller_region
    , l.logistics_tier AS seller_logistics_tier
FROM olist_maas_pipeline.bronze.sales_sellers s
LEFT JOIN olist_maas_pipeline.silver.dim_logistics_regions l 
    ON s.seller_state = l.state_code;

-- marketing growth master for seller acquisition analysis
CREATE OR REPLACE TABLE olist_maas_pipeline.silver.marketing_funnel AS
SELECT 
    m.mql_id
    , d.seller_id
    , m.origin
    , m.first_contact_date
    , d.won_date
    , d.business_segment
    , d.lead_type
    , d.lead_behaviour_profile
    , d.declared_monthly_revenue
FROM olist_maas_pipeline.silver.marketing_mql m
LEFT JOIN olist_maas_pipeline.bronze.marketing_closed_deals d 
    ON m.mql_id = d.mql_id;

-- customer master for CLV & persona analysis
CREATE OR REPLACE TABLE olist_maas_pipeline.silver.sales_customers AS
SELECT 
    c.customer_id
    , c.customer_unique_id
    , c.customer_zip_code_prefix
    , c.customer_city
    , c.customer_state
    , l.region_name
    , l.logistics_tier
FROM olist_maas_pipeline.bronze.sales_customers c
LEFT JOIN olist_maas_pipeline.silver.dim_logistics_regions l 
    ON c.customer_state = l.state_code;

-- geolocation master
CREATE OR REPLACE TABLE olist_maas_pipeline.silver.sales_geolocation AS
SELECT 
    geolocation_zip_code_prefix
    , AVG(geolocation_lat) AS latitude
    , AVG(geolocation_lng) AS longitude
    , FIRST(geolocation_city) AS city
    , FIRST(geolocation_state) AS state_code
FROM olist_maas_pipeline.bronze.sales_geolocation
WHERE (geolocation_lat BETWEEN -35 AND 6)
  AND (geolocation_lng BETWEEN -75 AND -33)
GROUP BY 1;